In [28]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

df = pd.read_csv('winequality-red.csv')
display(df.head(5))
print(df.columns)

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


Index(['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
       'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density',
       'pH', 'sulphates', 'alcohol', 'quality'],
      dtype='object')


In [29]:
display(df.isnull().sum())
display(df.nunique)
df.shape

fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
dtype: int64

<bound method DataFrame.nunique of       fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
0               7.4             0.700         0.00             1.9      0.076   
1               7.8             0.880         0.00             2.6      0.098   
2               7.8             0.760         0.04             2.3      0.092   
3              11.2             0.280         0.56             1.9      0.075   
4               7.4             0.700         0.00             1.9      0.076   
...             ...               ...          ...             ...        ...   
1594            6.2             0.600         0.08             2.0      0.090   
1595            5.9             0.550         0.10             2.2      0.062   
1596            6.3             0.510         0.13             2.3      0.076   
1597            5.9             0.645         0.12             2.0      0.075   
1598            6.0             0.310         0.47             3.6      0.

(1599, 12)

In [30]:
display(df.info())
display(df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1599 entries, 0 to 1598
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1599 non-null   float64
 1   volatile acidity      1599 non-null   float64
 2   citric acid           1599 non-null   float64
 3   residual sugar        1599 non-null   float64
 4   chlorides             1599 non-null   float64
 5   free sulfur dioxide   1599 non-null   float64
 6   total sulfur dioxide  1599 non-null   float64
 7   density               1599 non-null   float64
 8   pH                    1599 non-null   float64
 9   sulphates             1599 non-null   float64
 10  alcohol               1599 non-null   float64
 11  quality               1599 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 150.0 KB


None

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
count,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000
mean,8.319637,0.527821,0.270976,2.538806,0.087467,15.874922,46.467792,0.996747,3.311113,0.658149,10.422983,5.636023
std,1.741096,0.179060,0.194801,1.409928,0.047065,10.460157,32.895324,0.001887,0.154386,0.169507,1.065668,0.807569
min,4.600000,0.120000,0.000000,0.900000,0.012000,1.000000,6.000000,0.990070,2.740000,0.330000,8.400000,3.000000
25%,7.100000,0.390000,0.090000,1.900000,0.070000,7.000000,22.000000,0.995600,3.210000,0.550000,9.500000,5.000000
50%,7.900000,0.520000,0.260000,2.200000,0.079000,14.000000,38.000000,0.996750,3.310000,0.620000,10.200000,6.000000
75%,9.200000,0.640000,0.420000,2.600000,0.090000,21.000000,62.000000,0.997835,3.400000,0.730000,11.100000,6.000000
max,15.900000,1.580000,1.000000,15.500000,0.611000,72.000000,289.000000,1.003690,4.010000,2.000000,14.900000,8.000000


In [31]:
num_cols = df.select_dtypes(include=['int64', 'float64']).columns

for col in num_cols:
    print(f"{col}: Skewness = {df[col].skew():.2f}")

fixed acidity: Skewness = 0.98
volatile acidity: Skewness = 0.67
citric acid: Skewness = 0.32
residual sugar: Skewness = 4.54
chlorides: Skewness = 5.68
free sulfur dioxide: Skewness = 1.25
total sulfur dioxide: Skewness = 1.52
density: Skewness = 0.07
pH: Skewness = 0.19
sulphates: Skewness = 2.43
alcohol: Skewness = 0.86
quality: Skewness = 0.22


In [32]:
from sklearn.preprocessing import PowerTransformer

num_cols = df.select_dtypes(include=['int64', 'float64']).columns

skewed_cols = [col for col in num_cols if abs(df[col].skew()) > 0.5]

pt = PowerTransformer(method='yeo-johnson')
df[skewed_cols] = pt.fit_transform(df[skewed_cols])

for col in skewed_cols:
    print(f"{col}: Skewness (after Yeo-Johnson) = {df[col].skew():.2f}")

fixed acidity: Skewness (after Yeo-Johnson) = 0.00
volatile acidity: Skewness (after Yeo-Johnson) = 0.00
residual sugar: Skewness (after Yeo-Johnson) = -0.02
chlorides: Skewness (after Yeo-Johnson) = -0.15
free sulfur dioxide: Skewness (after Yeo-Johnson) = -0.01
total sulfur dioxide: Skewness (after Yeo-Johnson) = -0.00
sulphates: Skewness (after Yeo-Johnson) = 0.01
alcohol: Skewness (after Yeo-Johnson) = 0.11


In [33]:
for col in num_cols:
    df[col] = df[col].astype(float)  
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    before_outliers = ((df[col] < lower) | (df[col] > upper)).sum()

    df.loc[df[col] < lower, col] = lower
    df.loc[df[col] > upper, col] = upper
    print(f"{col}:= {before_outliers}")

fixed acidity:= 18
volatile acidity:= 7
citric acid:= 1
residual sugar:= 41
chlorides:= 118
free sulfur dioxide:= 0
total sulfur dioxide:= 2
density:= 45
pH:= 35
sulphates:= 13
alcohol:= 0
quality:= 28


In [34]:
num_cols = df.select_dtypes(include=['int64', 'float64']).columns

for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df.loc[df[col] < lower, col] = lower
    df.loc[df[col] > upper, col] = upper

    after_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f"{col}: Outliers after rectification = {after_outliers}")

fixed acidity: Outliers after rectification = 0
volatile acidity: Outliers after rectification = 0
citric acid: Outliers after rectification = 0
residual sugar: Outliers after rectification = 0
chlorides: Outliers after rectification = 0
free sulfur dioxide: Outliers after rectification = 0
total sulfur dioxide: Outliers after rectification = 0
density: Outliers after rectification = 0
pH: Outliers after rectification = 0
sulphates: Outliers after rectification = 0
alcohol: Outliers after rectification = 0
quality: Outliers after rectification = 0


In [35]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop('quality', axis=1)
y = df['quality']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [36]:
from sklearn.model_selection import GridSearchCV
from sklearn.neural_network import MLPRegressor
from sklearn.exceptions import ConvergenceWarning
import warnings

warnings.filterwarnings("ignore", category=ConvergenceWarning)

param_grid = {
    'hidden_layer_sizes': [(32, 16), (64, 32), (64, 32, 16)],
    'activation': ['relu', 'tanh'],
    'alpha': [0.0001, 0.001, 0.01],
    'learning_rate_init': [0.001, 0.01]
}

mlp = MLPRegressor(max_iter=2000, random_state=42)
grid = GridSearchCV(mlp, param_grid, cv=5, scoring='r2', n_jobs=-1)
grid.fit(X_train_scaled, y_train)

print("Best R2:", grid.best_score_)
print("Best Params:", grid.best_params_)


Best R2: 0.3375080105113454
Best Params: {'activation': 'tanh', 'alpha': 0.01, 'hidden_layer_sizes': (64, 32), 'learning_rate_init': 0.001}


In [37]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

reg = MLPRegressor(hidden_layer_sizes=(64, 32), activation='tanh', alpha=0.01, learning_rate_init=0.001)
reg.fit(X_train_scaled, y_train)

y_pred = reg.predict(X_test_scaled)

print("Mean Squared Error:", mean_squared_error(y_test, y_pred))
print("Mean Absolute Error:", mean_absolute_error(y_test, y_pred))
print("R2 Score:", r2_score(y_test, y_pred))

print(pd.DataFrame({'Actual': y_test[:10], 'Predicted': y_pred[:10]}))

Mean Squared Error: 0.3685128194589751
Mean Absolute Error: 0.48608009722765394
R2 Score: 0.3994093248141991
      Actual  Predicted
803      6.0   5.732872
124      5.0   4.888430
350      6.0   5.465744
682      5.0   5.529793
1326     6.0   6.015113
976      5.0   5.177710
1493     5.0   5.156737
706      5.0   5.027847
613      5.0   6.014827
1587     6.0   5.885421


In [38]:
import joblib
joblib.dump(reg, "wine_ann_model.joblib")
joblib.dump(scaler, "wine_scaler.joblib")


['wine_scaler.joblib']

In [39]:
# Uninstall current numpy
!pip uninstall -y numpy

# Install numpy 1.26.4
!pip install numpy==1.26.4

# (Optional) Restart the kernel to ensure change takes effect


Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4


You can safely remove it manually.
You can safely remove it manually.


  Using cached numpy-1.26.4-cp310-cp310-win_amd64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp310-cp310-win_amd64.whl (15.8 MB)


In [40]:
import numpy; print(numpy.__version__)
import sklearn; print(sklearn.__version__)


1.26.4
1.7.2


In [41]:
feature_names = [
    "fixed acidity", "volatile acidity", "citric acid", "residual sugar", "chlorides",
    "free sulfur dioxide", "total sulfur dioxide", "density", "pH", "sulphates", "alcohol"
]
sample_indices = X_test.sample(5, random_state=1).index
sample_features = X_test.loc[sample_indices]
sample_scaled = scaler.transform(sample_features)

# 5. Predict
sample_preds = reg.predict(sample_scaled)

# 6. Display sample readings and predictions
result_df = pd.DataFrame(sample_features, columns=feature_names)
result_df['Actual Quality'] = y_test.loc[sample_indices].values
result_df['Predicted Quality'] = sample_preds
print(result_df)

      fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
427        0.808509          1.357557         0.22       -0.597145  -0.120630   
588       -2.665994         -0.552983         0.24       -0.380998  -1.151624   
781       -1.177969         -0.299546         0.14        0.299203   1.283118   
1252      -0.670735          1.088853         0.00       -0.837193   1.507391   
865       -0.592974          0.678295         0.07        0.556565  -0.120630   

      free sulfur dioxide  total sulfur dioxide   density     pH  sulphates  \
427             -1.107886             -0.210431  0.998800  3.260  -0.581650   
588              0.557920              0.428834  0.992248  3.685   0.765223   
781             -0.555712             -0.003984  0.997320  3.660   0.183772   
1252            -1.107886             -1.351508  0.996270  3.450  -0.392318   
865              0.291904              1.222260  0.997480  3.510  -0.783598   

       alcohol  Actual Quality  Predic

In [42]:
row_num = 472
features_row = df.loc[row_num, feature_names].values.reshape(1, -1)

# Scale features
features_row_scaled = scaler.transform(features_row)

# Predict
predicted_quality = reg.predict(features_row_scaled)[0]

# Show result
print(f"True Quality: {df.loc[row_num, 'quality']}")
print(f"Predicted Quality: {predicted_quality:.4f}")
print("Feature values:", features_row.flatten())

True Quality: 6.0
Predicted Quality: 6.1076
Feature values: [ 1.95408665 -0.88823292  0.55        0.55656521  0.17399256  0.99385222
  0.87665586  0.9995      3.15        1.16887833  0.19325688]


c:\Users\NISHAL\anaconda3\envs\venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [43]:
row_num = 472
print(df.loc[row_num, feature_names].values)


[ 1.95408665 -0.88823292  0.55        0.55656521  0.17399256  0.99385222
  0.87665586  0.9995      3.15        1.16887833  0.19325688]


In [44]:
print(df.loc[472, feature_names].values)


[ 1.95408665 -0.88823292  0.55        0.55656521  0.17399256  0.99385222
  0.87665586  0.9995      3.15        1.16887833  0.19325688]


In [45]:
vals = df.loc[472, feature_names].values.reshape(1, -1)
pred = reg.predict(scaler.transform(vals))[0]
print(pred)


6.107592419785578


c:\Users\NISHAL\anaconda3\envs\venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [47]:
# Notebook cell before saving
import os
print("Saving model to:", os.path.abspath("wine_ann_model.joblib"))
print("Saving scaler to:", os.path.abspath("wine_scaler.joblib"))
joblib.dump(reg, "wine_ann_model.joblib")
joblib.dump(scaler, "wine_scaler.joblib")


Saving model to: c:\Users\NISHAL\OneDrive\Documents\Wine-ANN new\Wine_ANN_new\wine_ann_model.joblib
Saving scaler to: c:\Users\NISHAL\OneDrive\Documents\Wine-ANN new\Wine_ANN_new\wine_scaler.joblib


['wine_scaler.joblib']